# Textzusammenfassung

Jeder Artikel der Süddeutschen Zeitung in unserem Korpus hat zwei Fassungen: den vollen Text von durchschnittlich fast 480 Wörtern und einen **Teaser** von rund 22 Wörtern, den die Redaktion darübergesetzt hat. Der Teaser ist eine von Menschen geschriebene Zusammenfassung. Genau deshalb wurde das Korpus gesammelt: MLSUM (Scialom et al. 2020) ist ein Datensatz zum Trainieren und Bewerten von Verfahren der **automatischen Textzusammenfassung**.

Han, Pei und Tong (2023, Kap. 12.1) unterscheiden zwei grundsätzliche Ansätze:

- **Extraktive Zusammenfassung** wählt die wichtigsten Sätze des Originals aus und übernimmt sie unverändert.
- **Abstraktive Zusammenfassung** bildet eine interne Repräsentation des Inhalts und formuliert daraus einen neuen Text - so, wie es ein Mensch tun würde.

Dazwischen liegen **Extract-then-Rewrite**-Verfahren, die zuerst Sätze auswählen und sie dann umformulieren, und die **maschinengestützte Zusammenfassung**, bei der ein Mensch maschinelle Vorschläge überarbeitet. Wir fassen einzelne Dokumente zusammen (*single-document summarization*); Übung 4 geht zu mehreren Dokumenten über.

Am Ende dieses Skripts kannst du:

- Zusammenfassungen mit **ROUGE** gegen eine Referenz bewerten und die Grenzen dieses Maßes benennen;
- prüfen, ob die **Referenzdaten** tun, was sie sollen - hier tun sie es für ein Drittel der Artikel nicht;
- **extraktive Verfahren** bauen und vergleichen: die ersten Sätze (*Lead*), TF-IDF- und Embedding-Zentroide und **TextRank**, ein Graphverfahren nach dem Vorbild von PageRank;
- mit zwei **LLMs** (Claude und Mistral) abstraktiv und nach dem Extract-then-Rewrite-Prinzip zusammenfassen und das Ergebnis auf **erfundene Fakten** prüfen.

### Ausführen dieses Skripts

Der letzte Teil ruft zwei LLMs über ihre APIs auf: Claude (Anthropic) mit dem Paket `anthropic` und Mistral mit dem Paket `mistralai`. Die Schlüssel stehen in der Datei **`api-keys.env`** im Ordner des Kurses; eine Vorlage ohne Schlüssel liegt als `api-keys.example.env` daneben. `tm.load_api_keys()` liest die Datei und setzt die Umgebungsvariablen `ANTHROPIC_API_KEY` und `MISTRAL_API_KEY`. Ein Modell, für das kein Schlüssel vorhanden ist, wird übersprungen. Wird ein Schlüssel in der Datei ausgetauscht, genügt es, die Zelle mit `tm.load_api_keys()` weiter unten erneut auszuführen.

Wie in Skript 07 wird **nur dann etwas an eine API geschickt, wenn in der folgenden Zelle `RUN_LLM = True` steht.** Die Ergebnisse werden in `_cache/` gespeichert; ein späterer Lauf liest sie von dort, auch mit `RUN_LLM = False`. Ein vollständiger Lauf fasst 100 Artikel mit zwei Modellen und zwei Prompt-Varianten zusammen; bei den Preisen zum Zeitpunkt des Schreibens kostet das insgesamt etwa ein bis zwei US-Dollar, den größten Teil davon Claude. Jeder Lauf gibt die verbrauchten Tokens aus.

In [ ]:
# Schalter für die API-Aufrufe: Bei False wird nichts an eine API geschickt
RUN_LLM = False  # Setze auf True, um die APIs aufzurufen (kostet Geld - siehe oben)

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Standardbibliothek
import os
import re
import time
from collections import Counter

# Datentransformation und Visualisierung
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Satzsegmentierung
import spacy

# TF-IDF und Sentence Transformer für die extraktiven Verfahren
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# Clients für die APIs von Anthropic und Mistral
import anthropic
from mistralai.client import Mistral

# Hilfsfunktionen des Kurses
import textmining as tm

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## Daten

Wir nehmen alle SZ-Artikel des Korpus und ergänzen den vollen Text aus `tm.load_bodies()`. Der Teaser steht in der Spalte `lead`. Für die Bewertung ziehen wir eine feste Zufallsstichprobe von 1.000 Artikeln; das reicht für stabile Mittelwerte und hält die Rechenzeit klein.

In [ ]:
# SZ-Artikel mit Teaser und vollem Text
corpus = tm.load_corpus()
bodies = tm.load_bodies()
sz = corpus[corpus["outlet"] == "Süddeutsche Zeitung"].copy()
sz["body"] = sz["doc_key"].map(bodies)
sz = sz[sz["body"].notna() & (sz["lead"] != "")]

# Eine feste Stichprobe von 1.000 Artikeln
sample = sz.sample(1000, random_state=0).reset_index(drop=True)
print(f"{len(sz):,} SZ-Artikel mit Teaser und Text, davon {len(sample):,} in der Stichprobe")
print(f"Wörter im Teaser: Median {sample['lead'].str.split().str.len().median():.0f}, "
      f"Wörter im Text: Median {sample['body'].str.split().str.len().median():.0f}")

Ein Teaser ist etwa zwanzigmal kürzer als sein Artikel. Extraktive Verfahren arbeiten mit Sätzen, also zerlegen wir jeden Text in Sätze. Ein leeres deutsches spaCy-Modell mit dem regelbasierten `sentencizer` genügt dafür: Es trennt an Satzzeichen und kennt die Abkürzungen des deutschen Tokenizers (*z. B.*, *Dr.*), sodass sie keinen Satz beenden. Sehr kurze Bruchstücke (weniger als vier Wörter, oft Zwischenüberschriften) lassen wir weg.

In [ ]:
# Satzsegmentierung mit einem leeren deutschen Modell
segmenter = spacy.blank("de")
segmenter.add_pipe("sentencizer")
sample["sentences"] = [[s.text.strip() for s in doc.sents if len(s.text.split()) >= 4]
                       for doc in segmenter.pipe(sample["body"])]
print(f"Sätze je Artikel: Median {sample['sentences'].str.len().median():.0f}")
print("\nTeaser:", sample.loc[0, "lead"])
print("Erste Sätze des Artikels:", *sample.loc[0, "sentences"][:3], sep="\n  ")

Der erste "Satz" dieses Artikels ist eine Bildunterschrift (*… gibt ein Statement ab*), die beim Export der Daten in den Text geraten ist - in Skript 01 war das der Grund, den SZ-Text nicht als Teaser zu verwenden (Entscheidung 8). Und der Teaser selbst steht wörtlich in den Sätzen 2 und 3. Beides wird gleich wichtig.

## Ein Maß für Zusammenfassungen: ROUGE

Wie gut ist eine Zusammenfassung? Die Standardantwort der Forschung ist **ROUGE** (*Recall-Oriented Understudy for Gisting Evaluation*, Lin 2004): Je mehr Wörter und Wortfolgen eine Zusammenfassung mit einer Referenz teilt, desto besser.

- **ROUGE-1** vergleicht einzelne Wörter, **ROUGE-2** Bigramme (Paare aufeinanderfolgender Wörter).
- **ROUGE-L** sucht die längste gemeinsame Teilfolge (engl. *longest common subsequence*): die längste Folge von Wörtern, die in beiden Texten in derselben Reihenfolge vorkommt, aber nicht unbedingt direkt hintereinander.

Aus der Überlappung berechnen wir Precision (welcher Anteil der Zusammenfassung steht in der Referenz?), Recall (welcher Anteil der Referenz steht in der Zusammenfassung?) und deren harmonisches Mittel, den F1-Wert. Es gibt Pakete für ROUGE, aber die meisten sind für Englisch gebaut (mit englischem Stemming). Die Berechnung ist kurz genug, um sie selbst zu schreiben - und dann wissen wir genau, was gemessen wird:

In [ ]:
TOKEN = re.compile(r"\w+")


# Kleinbuchstaben, nur Wörter und Zahlen
def tokens(text: str) -> list[str]:
    return TOKEN.findall(text.lower())


# F1 aus Überlappung, Länge der Zusammenfassung und Länge der Referenz
def f1(overlap: int, n_summary: int, n_reference: int) -> float:
    if overlap == 0:
        return 0.0
    precision, recall = overlap / n_summary, overlap / n_reference
    return 2 * precision * recall / (precision + recall)


# ROUGE-N: Überlappung der N-Gramme (jedes N-Gramm zählt so oft, wie es in beiden vorkommt)
def rouge_n(summary: str, reference: str, n: int) -> float:
    s, r = tokens(summary), tokens(reference)
    s_grams, r_grams = Counter(zip(*[s[i:] for i in range(n)])), Counter(zip(*[r[i:] for i in range(n)]))
    return f1(sum((s_grams & r_grams).values()), sum(s_grams.values()), sum(r_grams.values()))


# ROUGE-L: Länge der längsten gemeinsamen Teilfolge (dynamische Programmierung)
def rouge_l(summary: str, reference: str) -> float:
    s, r = tokens(summary), tokens(reference)
    previous = [0] * (len(r) + 1)
    for word in s:
        current = [0]
        for j, other in enumerate(r):
            current.append(previous[j] + 1 if word == other else max(previous[j + 1], current[j]))
        previous = current
    return f1(previous[-1], len(s), len(r))


def rouge(summary: str, reference: str) -> dict:
    return {"ROUGE-1": rouge_n(summary, reference, 1), "ROUGE-2": rouge_n(summary, reference, 2),
            "ROUGE-L": rouge_l(summary, reference)}


# Ein Beispiel: dieselbe Aussage, einmal umgestellt, einmal mit anderen Worten
reference = "Die Regierung beschließt eine Obergrenze für Flüchtlinge."
pd.DataFrame({s: rouge(s, reference) for s in [
    "Die Regierung beschließt eine Obergrenze für Flüchtlinge.",
    "Eine Obergrenze für Flüchtlinge beschließt die Regierung.",
    "Das Kabinett legt eine Höchstzahl für Asylbewerber fest.",
]}).T.round(2)

Das Beispiel zeigt in drei Zeilen, was ROUGE misst und was nicht. Die umgestellte Fassung behält ROUGE-1 von 1,0, verliert aber bei ROUGE-2 und ROUGE-L, weil sich die Wortfolge geändert hat. Die dritte Fassung sagt dasselbe mit anderen Worten und verliert stark: ROUGE-2 fällt auf null, ROUGE-1 und ROUGE-L auf 0,27 - gemeinsam sind nur *eine* und *für*. **ROUGE belohnt Übereinstimmung im Wortlaut, nicht in der Bedeutung.** Das bevorzugt extraktive Verfahren, die den Wortlaut des Originals übernehmen, und bestraft gute Paraphrasen - wir kommen darauf zurück, wenn die LLMs ins Spiel kommen.

## Eine Überraschung in den Referenzen

Bevor wir Verfahren vergleichen, prüfen wir die Referenzen. Die einfachste denkbare Zusammenfassung eines Nachrichtenartikels sind seine ersten Sätze: Journalisten schreiben nach dem Prinzip der *umgekehrten Pyramide*, das Wichtigste zuerst. Wie ähnlich ist der Teaser dem *ähnlichsten* Satz des Artikels?

In [ ]:
# ROUGE-2 des Satzes, der dem Teaser am ähnlichsten ist, und Position dieses Satzes
best = [max(((rouge_n(s, ref, 2), i) for i, s in enumerate(sents)), default=(0.0, -1))
        for sents, ref in zip(sample["sentences"], sample["lead"])]
sample["best_r2"], sample["best_position"] = [b[0] for b in best], [b[1] for b in best]

# Verteilung als Histogramm
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(sample["best_r2"], bins=40, color="#2F6690")
ax.axvline(0.5, color="0.3", linestyle="--", linewidth=1)
ax.text(0.51, ax.get_ylim()[1] * 0.9, "Teaser steht im Text", fontsize=8, color="0.3")
ax.set(xlabel="ROUGE-2 zwischen Teaser und dem ähnlichsten Satz des Artikels", ylabel="Artikel")
plt.show()
copied = sample["best_r2"] >= 0.5
print(f"{copied.mean():.0%} der Artikel enthalten einen Satz mit ROUGE-2 >= 0,5 zum Teaser")
print("Position dieses Satzes:", sample.loc[copied, "best_position"].value_counts().head(3).to_dict())

Die Verteilung hat zwei Gipfel. Bei den meisten Artikeln ist auch der ähnlichste Satz weit vom Teaser entfernt; der Teaser ist ein eigener Text. Bei einem knappen Drittel aber steht der Teaser **fast wörtlich im Artikel**, fast immer in einem der ersten drei Sätze. Beim Export der Daten ist der Teaser offenbar in den Text geraten.

In [ ]:
# Zwei Beispiele für Artikel, deren Teaser im Text steht
for r in sample[copied].head(2).itertuples():
    print("Teaser:     ", r.lead)
    print("Satz", r.best_position + 1, "im Text:", r.sentences[r.best_position], "\n")

Für die Bewertung ist das fatal: Auf diesen Artikeln muss ein Verfahren nur einen der ersten Sätze abschreiben, um eine nahezu perfekte Punktzahl zu bekommen. Wer sie mitbewertet, misst vor allem, wie oft der Teaser im Text steht - eine Form von **Data Leakage**, wie wir sie aus Skript 05 kennen, nur steckt sie diesmal in den Referenzen. Wir bewerten deshalb nur auf den Artikeln, in denen der Teaser *nicht* im Text steht.

In [ ]:
# Nur Artikel, deren Teaser nicht (fast) wörtlich im Text steht
clean = sample[~copied].reset_index(drop=True)
print(f"{len(clean)} Artikel für die Bewertung")

## Extraktive Verfahren

Alle extraktiven Verfahren funktionieren gleich: Sie geben jedem Satz eine Punktzahl, sortieren die Sätze danach und nehmen so viele der besten, bis das **Längenbudget** von 25 Wörtern erreicht ist - etwa die Länge eines Teasers. Der letzte Satz wird dabei notfalls mitten im Satz abgeschnitten. Die ausgewählten Sätze werden in ihrer ursprünglichen Reihenfolge ausgegeben. Weil ROUGE längere Zusammenfassungen beim Recall bevorzugt, ist ein gleiches Budget für alle Verfahren die Voraussetzung für einen fairen Vergleich.

Die Verfahren unterscheiden sich nur darin, wie sie die Sätze ordnen:

- **Lead**: die Reihenfolge im Text. Eine starke Baseline für Nachrichten.
- **Zufall**: eine zufällige Reihenfolge. Die Untergrenze.
- **TF-IDF-Zentroid**: Der Mittelwert aller Satzvektoren ist der "Schwerpunkt" des Artikels; wichtig ist ein Satz, der ihm nahe liegt.
- **Embedding-Zentroid**: dasselbe mit den Sentence-Transformer-Embeddings aus Skript 06.
- **TextRank** (Mihalcea und Tarau 2004): Die Sätze bilden einen Graphen, verbunden nach ihrer Ähnlichkeit. Wichtig ist ein Satz, der vielen anderen ähnlich ist, die ihrerseits wichtig sind. Das ist die Idee von **PageRank**, angewandt auf Sätze statt Webseiten.
- **Orakel**: die Sätze, die der Referenz einzeln am ähnlichsten sind. Es "schummelt", weil es die Referenz kennt, und zeigt damit ungefähr, was eine extraktive Zusammenfassung erreichen kann. Eine echte Obergrenze ist es nicht: Es bewertet jeden Satz für sich (*greedy*), nicht die beste Kombination von Sätzen.

In [ ]:
BUDGET = 25  # Wörter je Zusammenfassung


# Nimm Sätze in der Reihenfolge `order`, bis das Budget erreicht ist; der letzte Satz wird
# notfalls mitten im Satz abgeschnitten. Erst danach kommen die Sätze in Textreihenfolge.
def extract(sentences: list[str], order) -> str:
    chosen, words = {}, 0
    for i in order:
        part = sentences[i].split()[: BUDGET - words]
        chosen[i] = " ".join(part)
        words += len(part)
        if words >= BUDGET:
            break
    return " ".join(chosen[i] for i in sorted(chosen))


# Die IDF-Gewichte lernen wir aus SZ-Artikeln, die nicht in der Stichprobe sind
tfidf = TfidfVectorizer(sublinear_tf=True).fit(sz.loc[~sz["doc_key"].isin(sample["doc_key"]), "body"])


def centroid_order(X) -> np.ndarray:
    centre = np.asarray(X.mean(axis=0)).ravel()
    return np.argsort(-(X @ centre))


def textrank_order(X, damping: float = 0.85, iterations: int = 50) -> np.ndarray:
    # Ähnlichkeitsgraph der Sätze (Kosinus, ohne Selbstschleifen), zeilenweise normiert
    S = np.asarray((X @ X.T).todense()) if hasattr(X, "todense") else X @ X.T
    np.fill_diagonal(S, 0)
    S = S / np.maximum(S.sum(axis=1, keepdims=True), 1e-12)
    # PageRank durch Potenziteration
    n = len(S)
    rank = np.full(n, 1 / n)
    for _ in range(iterations):
        rank = (1 - damping) / n + damping * S.T @ rank
    return np.argsort(-rank)

Für den Embedding-Zentroid kodieren wir alle Sätze der Stichprobe einmal mit dem Sentence Transformer aus Skript 06:

In [ ]:
# Sentence Transformer aus Skript 06; Grafikkarte, falls vorhanden
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)

# Kodiere alle Sätze auf einmal und teile die Matrix danach wieder auf die Artikel auf
all_sentences = [s for sents in clean["sentences"] for s in sents]
E = encoder.encode(all_sentences, batch_size=256, normalize_embeddings=True, show_progress_bar=False)
bounds = np.cumsum([0] + clean["sentences"].str.len().tolist())
clean["embeddings"] = [E[a:b] for a, b in zip(bounds[:-1], bounds[1:])]
print(f"{len(all_sentences):,} Sätze kodiert auf {device}")

In [ ]:
rng = np.random.default_rng(0)
METHODS = {
    "Lead": lambda r: range(len(r.sentences)),
    "Zufall": lambda r: rng.permutation(len(r.sentences)),
    "TF-IDF-Zentroid": lambda r: centroid_order(tfidf.transform(r.sentences)),
    "Embedding-Zentroid": lambda r: centroid_order(r.embeddings),
    "TextRank": lambda r: textrank_order(tfidf.transform(r.sentences)),
    "Orakel": lambda r: np.argsort([-rouge_n(s, r.lead, 2) - rouge_n(s, r.lead, 1) for s in r.sentences]),
}

# Erzeuge mit jedem Verfahren eine Zusammenfassung pro Artikel
summaries = pd.DataFrame({name: [extract(r.sentences, order(r)) if r.sentences else "" for r in clean.itertuples()]
                          for name, order in METHODS.items()})


# Mittelwerte von ROUGE über alle Artikel
def score(texts, references) -> pd.Series:
    return pd.DataFrame([rouge(t, ref) for t, ref in zip(texts, references)]).mean()


results = pd.DataFrame({name: score(summaries[name], clean["lead"]) for name in summaries}).T
results.round(3)

Drei Beobachtungen:

- **Die Werte sind niedrig, auch für das Orakel.** Selbst die Sätze, die dem Teaser einzeln am ähnlichsten sind, teilen nur einen kleinen Teil ihrer Bigramme mit ihm. Der Teaser ist kein Auszug, sondern ein eigener Text: Er spitzt zu, stellt einen Bezug her, kündigt an (*Was hinter dem Streit steckt*). Ein extraktives Verfahren kann das grundsätzlich nicht.
- **Lead ist kaum zu schlagen.** Die ersten Sätze eines Nachrichtenartikels sind schon eine Zusammenfassung; die umgekehrte Pyramide ist journalistisches Handwerk. TextRank und der TF-IDF-Zentroid liegen etwas darunter, klar über dem Zufall. Mit der richtigen Hypothese über die Textsorte ist eine Zeile Code eine starke Baseline - wie die häufigste Klasse in Skript 05.
- **Embeddings helfen hier nicht.** Der Embedding-Zentroid ist sogar schlechter als der TF-IDF-Zentroid und liegt bei ROUGE-2 nicht weit über dem Zufall. Für die Frage "welcher Satz liegt in der Mitte des Artikels?" reicht die Wortüberlappung; ROUGE misst ohnehin Wortüberlappung.

Wie sehen die Zusammenfassungen aus? Ein Beispiel:

In [ ]:
# Ein Artikel mit Teaser und den Zusammenfassungen der Verfahren
i = 7
print("Schlagzeile:", clean.loc[i, "title"])
print("Teaser:     ", clean.loc[i, "lead"], "\n")
for name in ["Lead", "TextRank", "Embedding-Zentroid", "Orakel"]:
    print(f"{name:19}", summaries.loc[i, name])

## Abstraktive Zusammenfassung mit LLMs

Ein LLM kann tun, was kein extraktives Verfahren kann: einen neuen Text schreiben. Wir vergleichen zwei Modelle verschiedener Anbieter - **Claude** von Anthropic und **Mistral** vom gleichnamigen französischen Anbieter - und zwei Prompt-Varianten:

- **Abstraktiv**: Das Modell bekommt den ganzen Artikel und soll einen Teaser schreiben.
- **Extract-then-Rewrite**: Das Modell bekommt nur die fünf besten Sätze nach TextRank und soll daraus einen Teaser formulieren. Das ist billiger (weniger Input), und das Modell kann nur umformulieren, was die Extraktion ausgewählt hat.

Weil jeder Aufruf Geld kostet, beschränken wir uns auf die ersten 100 Artikel der bereinigten Stichprobe. Die extraktiven Verfahren bewerten wir für den Vergleich auf denselben 100 Artikeln neu.

Den Prompt schreiben wir auf Deutsch und geben ihm das Längenbudget mit. Ein Hinweis auf den Stil eines Teasers gehört dazu: Ohne ihn schreiben LLMs gern eine neutrale Inhaltsangabe (*Der Artikel behandelt …*).

In [ ]:
# Die 100 Artikel für die LLMs
llm_sample = clean.head(100).copy()

SYSTEM = (
    "Du schreibst Teaser für Artikel einer deutschen Tageszeitung. Ein Teaser steht unter der "
    "Schlagzeile, fasst den Kern des Artikels in ein bis zwei Sätzen zusammen und macht neugierig. "
    f"Schreibe höchstens {BUDGET} Wörter. Gib nur den Teaser aus, ohne Anführungszeichen und ohne Einleitung. "
    "Verwende nur Informationen aus dem gegebenen Text."
)


def prompt_full(r) -> str:
    return f"Schlagzeile: {r.title}\n\nArtikel:\n{r.body}"


def prompt_extract(r) -> str:
    top = sorted(textrank_order(tfidf.transform(r.sentences))[:5])
    return f"Schlagzeile: {r.title}\n\nDie wichtigsten Sätze des Artikels:\n" + "\n".join(r.sentences[i] for i in top)


print(SYSTEM, "\n")
print(prompt_extract(next(llm_sample.itertuples())))

Jetzt die Clients. `tm.load_api_keys()` liest `api-keys.env` und meldet, welche Schlüssel vorhanden sind (die Schlüssel selbst werden nie ausgegeben). Für Claude verwenden wir dasselbe Modell und dieselben Einstellungen wie in Skript 07 (nur `max_tokens` ist kleiner, weil ein Teaser kurz ist); für Mistral das jeweils aktuelle große Modell (`mistral-large-latest` ist ein Alias, der auf die neueste Version zeigt). Die Temperatur setzen wir bei Mistral auf 0, damit ein zweiter Lauf möglichst dasselbe Ergebnis liefert.

In [ ]:
# Lies die Schlüssel aus api-keys.env
keys = tm.load_api_keys()
print(keys)

CLAUDE_MODEL = "claude-opus-5-5"
MISTRAL_MODEL = "mistral-large-latest"
# Das SDK von Anthropic wiederholt Anfragen bei Überlastung selbst; wir erlauben mehr Versuche als üblich
claude = anthropic.Anthropic(max_retries=6) if keys["ANTHROPIC_API_KEY"] else None
mistral = Mistral(api_key=os.environ["MISTRAL_API_KEY"]) if keys["MISTRAL_API_KEY"] else None


# Ein Teaser von Claude (gleiche Einstellungen wie in Skript 07, aber ohne strukturierte Ausgabe)
def ask_claude(prompt: str) -> tuple[str | None, int, int]:
    response = claude.beta.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        output_config={"effort": "low"},
        system=SYSTEM,
        messages=[{"role": "user", "content": prompt}],
    )
    if response.stop_reason != "end_turn":
        print(f"  keine Antwort: stop_reason={response.stop_reason}")
        return None, response.usage.input_tokens, response.usage.output_tokens
    text = [block.text for block in response.content if block.type == "text"][-1]
    return text.strip(), response.usage.input_tokens, response.usage.output_tokens


# Ein Teaser von Mistral; bei Überlastung oder Ratenlimit (viele Teilnehmende teilen einen Schlüssel)
# wird die Anfrage nach einer Pause wiederholt
def ask_mistral(prompt: str, attempts: int = 5) -> tuple[str | None, int, int]:
    for attempt in range(attempts):
        try:
            response = mistral.chat.complete(
                model=MISTRAL_MODEL,
                temperature=0,
                max_tokens=200,
                messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}],
            )
            break
        except Exception as error:  # z. B. HTTP 429 (zu viele Anfragen)
            if attempt == attempts - 1:
                print(f"  keine Antwort: {error}")
                return None, 0, 0
            time.sleep(2 ** attempt * 5)
    content = response.choices[0].message.content
    # Meist ein String; manche Modelle liefern eine Liste von Teilen (Text, Denkschritte)
    if not isinstance(content, str):
        content = "".join(getattr(part, "text", "") for part in content or [])
    return content.strip(), response.usage.prompt_tokens, response.usage.completion_tokens

In [ ]:
# Fasse die 100 Artikel mit einem Modell und einem Prompt zusammen, oder lies frühere Ergebnisse aus _cache/
def run(model_name: str, ask, client, prompt_name: str, make_prompt) -> pd.Series | None:
    path = tm.CACHE / f"summary-{model_name}-{prompt_name}-{len(llm_sample)}.parquet"
    if path.exists():
        print(f"{model_name}/{prompt_name}: lese gespeicherte Ergebnisse")
        return pd.read_parquet(path)["summary"]
    if not RUN_LLM or client is None:
        print(f"{model_name}/{prompt_name}: übersprungen ({'kein Schlüssel' if client is None else 'RUN_LLM ist False'})")
        return None
    texts, tokens_in, tokens_out, start = [], 0, 0, time.time()
    for k, r in enumerate(llm_sample.itertuples()):
        text, n_in, n_out = ask(make_prompt(r))
        texts.append(text)
        tokens_in, tokens_out = tokens_in + n_in, tokens_out + n_out
        print(f"  {model_name}/{prompt_name}: {k + 1}/{len(llm_sample)}", end="\r")
    print(f"\n{model_name}/{prompt_name}: {time.time() - start:.0f}s, Tokens rein {tokens_in:,}, raus {tokens_out:,}")
    result = pd.DataFrame({"doc_key": llm_sample["doc_key"], "summary": texts})
    tm.CACHE.mkdir(exist_ok=True)
    result.to_parquet(path)
    return result["summary"]


llm = {}
for model_name, ask, client in [("claude", ask_claude, claude), ("mistral", ask_mistral, mistral)]:
    for prompt_name, make_prompt in [("abstraktiv", prompt_full), ("extract-rewrite", prompt_extract)]:
        result = run(model_name, ask, client, prompt_name, make_prompt)
        if result is not None:
            llm[f"{model_name}: {prompt_name}"] = result.fillna("").to_numpy()

Der Vergleich auf den 100 Artikeln. Neben ROUGE geben wir die mittlere Länge in Wörtern aus, weil ROUGE von der Länge abhängt:

In [ ]:
# Extraktive Zusammenfassungen derselben 100 Artikel und, falls vorhanden, die der LLMs
compare = {name: summaries.loc[llm_sample.index, name].to_numpy() for name in ["Lead", "TextRank", "Orakel"]}
compare.update(llm)
table = pd.DataFrame({name: score(texts, llm_sample["lead"]) for name, texts in compare.items()}).T
table["Wörter"] = [np.mean([len(t.split()) for t in texts]) for texts in compare.values()]
table.round(3)

Wenn du die LLMs ausgeführt hast, lies die Tabelle mit dem ROUGE-Beispiel vom Anfang im Kopf. Ein abstraktiver Teaser formuliert neu und verliert dadurch bei ROUGE-2, selbst wenn er inhaltlich näher am Teaser der Redaktion ist als die ersten Sätze des Artikels. Ob ein LLM an das Orakel heranreicht, das die Referenz kennt, ist die interessantere Frage als der Abstand zu Lead. Und der Vergleich von *abstraktiv* und *Extract-then-Rewrite* zeigt, wie viel das Modell vom ganzen Artikel braucht.

ROUGE allein reicht für abstraktive Verfahren nicht. Lies die Teaser:

In [ ]:
# Die Teaser der Redaktion, von Lead und den LLMs für einige Artikel
for i in range(3):
    print("Schlagzeile:", llm_sample["title"].iloc[i])
    print(f"  {'Redaktion':28}", llm_sample["lead"].iloc[i])
    for name, texts in compare.items():
        if name != "Orakel":
            print(f"  {name:28}", texts[i])
    print()

## Erfundene Fakten

Ein extraktives Verfahren kann nichts erfinden: Jeder Satz stammt aus dem Artikel. Ein abstraktives Verfahren kann es. **Halluzinationen** - Fakten, die nicht im Original stehen - sind das größte Risiko abstraktiver Zusammenfassung, und ROUGE bemerkt sie nicht.

Eine einfache, automatische Prüfung: Namen und Zahlen sind die Fakten, auf die es in Nachrichten ankommt. Wir sammeln in jeder Zusammenfassung die großgeschriebenen Wörter, die nicht am Satzanfang stehen, und alle Zahlen, und prüfen, welcher Anteil davon **nicht im Artikel** vorkommt. Großgeschriebene Wörter sind im Deutschen auch gewöhnliche Substantive; ein neues Substantiv ist noch keine Halluzination, aber ein Hinweis darauf, dass das Modell neue Wörter eingeführt hat. Zum Vergleich messen wir dasselbe für den Teaser der Redaktion.

In [ ]:
# Namen, Substantive und Zahlen einer Zusammenfassung (ohne das erste Wort eines Satzes)
FACT = re.compile(r"(?<![.!?:] )(?<!^)\b(?:[A-ZÄÖÜ][\wäöüß-]+|\d[\d.,]*)")


# Anteil der Fakten-Wörter, die nicht im Artikel vorkommen
def novel_facts(summary: str, article: str) -> float:
    facts = FACT.findall(summary)
    return np.mean([f.rstrip(".,") not in article for f in facts]) if facts else 0.0


novelty = {"Redaktion (Teaser)": llm_sample["lead"].to_numpy()} | compare
pd.Series({name: np.mean([novel_facts(s, a) for s, a in zip(texts, llm_sample["body"])])
           for name, texts in novelty.items()}, name="Anteil neuer Namen/Substantive/Zahlen").round(3).to_frame()

Die extraktiven Verfahren liegen erwartungsgemäß bei null. Bemerkenswert ist die Zeile der Redaktion: **Auch die Teaser der SZ enthalten Wörter, die nicht im Artikel stehen** - fast ein Drittel ihrer Namen, Substantive und Zahlen - neue Begriffe, Bewertungen, Kontext, den die Redaktion für bekannt hält. Für einen Datensatz, mit dem Modelle das Zusammenfassen lernen, ist das ein Problem: Wer lernt, solche Referenzen nachzuahmen, lernt auch, Dinge hinzuzufügen, die nicht im Text stehen.

Für die LLMs zeigt der Wert, wie frei sie formulieren. Ein neues Substantiv ist harmlos, ein neuer Name oder eine neue Zahl nicht. Übung 2 sieht sich die Fälle einzeln an.

## Zusammenfassung

| Verfahren | Ansatz | Stärke | Schwäche |
|---|---|---|---|
| Lead | extraktiv, Position | starke Baseline für Nachrichten | nur für Textsorten mit umgekehrter Pyramide |
| TF-IDF- und Embedding-Zentroid | extraktiv, Nähe zum Schwerpunkt | unabhängig von der Position | bevorzugt lange, allgemeine Sätze |
| TextRank | extraktiv, Graph (PageRank) | findet zentrale Sätze ohne Training | ähnlich wie Zentroide; Sätze bleiben unverändert |
| Orakel | extraktiv, kennt die Referenz | Anhaltspunkt für das extraktiv Erreichbare | nur zur Bewertung; keine echte Obergrenze |
| LLM, abstraktiv | generativ | formuliert neu wie ein Mensch | kann Fakten erfinden; teuer |
| LLM, Extract-then-Rewrite | Auswahl + Umformulierung | weniger Input, Inhalt durch Extraktion begrenzt | Fehler der Extraktion bleiben |

Zwei Lehren gehen über die Zusammenfassung hinaus. **Prüfe die Referenzdaten**: In einem Drittel der Artikel stand der Teaser im Text, und ohne diese Prüfung hätten wir vor allem das gemessen. Und **prüfe, was das Maß misst**: ROUGE zählt gemeinsame Wörter. Für extraktive Verfahren ist das angemessen, für abstraktive nur ein grober Anhaltspunkt - die Bewertung von Zusammenfassungen durch Menschen oder durch ein zweites LLM ist ein eigenes Forschungsfeld.

## Übungen

1. **Das Budget.** Wiederhole den Vergleich der extraktiven Verfahren mit einem Budget von 15 und von 50 Wörtern. Ändert sich die Rangfolge? Warum steigen die Werte nicht einfach mit der Länge?
2. **Halluzinationen lesen.** Sammle aus den LLM-Teasern alle Namen und Zahlen, die nicht im Artikel stehen. Sortiere sie in drei Gruppen: harmlos (allgemeines Wissen, etwa ein Ländername), Umschreibung (*Kanzlerin* statt *Merkel*), erfunden. Unterscheiden sich die beiden Modelle?
3. **Ein anderer Textanfang.** Die SZ-Texte beginnen oft mit einer szenischen Einleitung (*Am Montagnachmittag, kurz vor halb fünf, …*). Bei welchen Ressorts (`section`) ist Lead stark, bei welchen schwach? Was sagt das über die Schreibweise der Ressorts?
4. **Mehrere Dokumente.** Fasse alle Artikel beider Zeitungen über ein Ereignis zusammen (etwa die Anschläge von Paris, siehe die Chronik in Skript 08): Wähle mit TextRank die zentralen Sätze über *alle* Artikel hinweg. Was musst du gegen Wiederholungen tun, wenn mehrere Artikel dasselbe sagen?
5. **Ein LLM als Gutachter.** Lass Claude für 20 Artikel beurteilen, welcher von zwei Teasern (Redaktion oder Mistral) den Artikel treffender zusammenfasst, ohne zu verraten, welcher von wem ist. Wie oft gewinnt die Redaktion? Vertausche die Reihenfolge der beiden Teaser: Ändert sich das Urteil?

## Quellen

* Han, J., Pei, J., Tong, H. (2023): *Data Mining: Concepts and Techniques*, 4. Aufl., Kap. 12.1. Morgan Kaufmann.
* Lin, C.-Y. (2004): ROUGE: A Package for Automatic Evaluation of Summaries. *Text Summarization Branches Out*, ACL Workshop, 74-81.
* Mihalcea, R., Tarau, P. (2004): TextRank: Bringing Order into Text. *Proceedings of EMNLP 2004*, 404-411.
* Scialom, T., Dray, P.-A., Lamprier, S., Piwowarski, B., Staiano, J. (2020): MLSUM: The Multilingual Summarization Corpus. *Proceedings of EMNLP 2020*, 8051-8067.
* https://platform.claude.com/docs
* https://docs.mistral.ai/
* https://sbert.net/